# 2단계 GraphSAGE 실험: 집계 방식 · 레이어 수 · 학습률

2단계 재료 기여도 모델(`IngredientLinkEncoder`)에서 **GraphSAGE 부분만 바꿔** 세 가지 실험을 합니다. 1단계 모델은 기존 체크포인트로 고정해 레시피 임베딩만 만듭니다. 학습 절차(Adam, 100 epoch 고정, 5-폴드 교대, 연결을 비운 그래프 손실 병행)와 지표(기여도 MAE·RMSE)는 `training/stage2.py train` + `training/evaluate.py`와 같습니다.

**준비:** Colab 파일 탭에서 아래 세 파일을 같은 경로로 올립니다(`/content` 기준). 경로가 다르면 [2] 공통 설정에서 바꾸세요.

| 파일 | 내용 |
|---|---|
| `data/stage1_rebuild_raw_gte.pt` | 레시피 GTE 임베딩 + 1단계 분할(SIMILAR 그래프) |
| `checkpoints/stage1_rebuild_raw_gte_h64.pt` | 고정할 1단계 모델 |
| `tmp/stage2_embeddings.pt` | 재료 GTE 임베딩 + 기여도 train/test 분할 (`python -m training.stage2 prepare` 결과, `colab_pipeline.ipynb`에서 만들었다면 `artifacts/stage2_embeddings.pt`) |

**장치:** GPU 런타임이면 GPU로, 아니면 CPU로 자동 학습합니다(`torch.cuda.is_available()`). CPU는 단일 스레드로 결정론적이고, GPU는 연산 순서가 달라 수치가 조금 다릅니다. 장치가 다른 결과는 이어 하기에서 섞이지 않으니 **한 실험 안의 조건 비교는 같은 장치에서** 하세요. GRU 조건은 CPU에서 1회 약 12분 걸리므로 GPU 런타임을 권장합니다.

**기존 2단계와 다른 점 두 가지**
1. **검증 분할:** 기존 2단계는 train/test만 있어 조건을 고르면 test가 선택에 쓰입니다. 그래서 학습 분할의 `VAL_RATIO`(기본 15%)를 검증 쌍으로 떼어 조건 선택에 쓰고, 테스트는 참고용으로만 봅니다. 검증 쌍은 학습 대상과 학습용 그래프에서 빠지고, 테스트 평가 그래프에는 들어갑니다(기존 평가처럼 test 외 모든 연결 사용). `VAL_RATIO = 0`이면 기존과 완전히 같은 절차이고, 같은 시드에서 기존 코드와 **비트 단위까지 같은 결과**가 나옵니다(이때는 테스트로 정렬).
2. **시드 고정:** 기존 `stage2 train`은 모델 초기화 시드를 고정하지 않아 실행마다 결과가 달라집니다. 여기서는 시드로 고정하고 시드 여러 개의 평균 ± 표준편차를 봅니다.

**실행:** [1] → [2]를 먼저 실행한 뒤 실험 셀을 순서대로 실행합니다. 각 실험 셀 위쪽 변수를 바꾸고 다시 실행하면 조건을 바꿀 수 있고, 이미 학습한 조건은 저장된 결과(`results/experiments_stage2/`)를 읽어 새 조건만 학습합니다. 표에서 1위와의 검증 RMSE 차이가 1위의 시드 표준편차보다 작은 조건에는 **≈**를 붙입니다. Colab 런타임이 끊기면 파일이 사라지니 마지막 셀로 내려받아 두세요.

In [ ]:
# ======================================================================
# [1] 공통 코드: 2단계 모델·학습 루프·평가 지표 (training/stage2.py train + training/evaluate.py와 동일 절차)
# ======================================================================
from __future__ import annotations

import copy
import csv
import hashlib
import json
import math
import os
import random
import statistics
from dataclasses import asdict, dataclass, replace
from pathlib import Path

# CUDA에서 결정론적 행렬곱(cuBLAS)에 필요: CUDA 초기화 전에 설정해야 함
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
from torch import nn
from torch.nn import functional as F

# GPU가 있으면 GPU, 없으면 CPU로 학습
# (CPU는 단일 스레드로 결정론적, GPU는 연산 순서가 달라 수치가 조금 다름)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STAGE1_DATA = Path("data/stage1_rebuild_raw_gte.pt")  # 레시피 GTE 임베딩 + 1단계 분할(SIMILAR 그래프)
STAGE1_CHECKPOINT = Path("checkpoints/stage1_rebuild_raw_gte_h64.pt")  # 고정할 1단계 모델
STAGE2_DATA = Path("tmp/stage2_embeddings.pt")  # 재료 GTE 임베딩 + 기여도 train/test 분할
RESULTS = Path("results/experiments_stage2")
AGGREGATORS = ("weighted_count", "weighted_mean", "mean", "max", "gru")
GRU_ORDERS = ("fixed", "shuffle", "weight")
VAL_SEED = 42  # 검증 분할 시드 (학습 시드와 무관하게 모든 실행이 같은 검증 쌍을 씀)


# ---------------- 데이터 ----------------


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def stage1_context_edges(rows):
    # 1단계 SIMILAR 연결 (training/stage1.py _context_edges와 동일)
    positive = [(a, b, score / 4) for a, b, score in rows if score >= 1]
    if not positive:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    forward = torch.tensor([(a, b) for a, b, _ in positive], dtype=torch.long).t()
    weights = torch.tensor([weight for _, _, weight in positive], dtype=torch.float32)
    return torch.cat((forward, forward.flip(0)), dim=1), weights.repeat(2)


class _Stage1SAGE(nn.Module):
    # 1단계 WeightedSAGELayer (src/engine.py)와 같은 계산
    def __init__(self, hidden: int):
        super().__init__()
        self.self_linear = nn.Linear(hidden, hidden)
        self.neighbor_linear = nn.Linear(hidden, hidden, bias=False)

    def forward(self, x, edges, weights):
        neighbors = torch.zeros_like(x)
        counts = x.new_zeros(len(x))
        if weights.numel():
            source, target = edges
            neighbors.index_add_(0, target, x[source] * weights[:, None])
            counts.index_add_(0, target, weights)
        mean = neighbors / counts.clamp_min(1e-8)[:, None]
        return self.self_linear(x) + self.neighbor_linear(mean)


class FrozenRecipeEncoder(nn.Module):
    # 1단계 RecipePairGraphSAGE의 레시피 인코더 부분 (평가 모드 forward와 같은 결과)
    def __init__(self, input_dim: int, hidden: int):
        super().__init__()
        self.input = nn.Sequential(nn.Linear(input_dim, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.sage1 = _Stage1SAGE(hidden)
        self.norm1 = nn.LayerNorm(hidden)
        self.sage2 = _Stage1SAGE(hidden)
        self.norm2 = nn.LayerNorm(hidden)

    def forward(self, features, edges, weights):
        x = F.gelu(self.norm1(self.sage1(self.input(features), edges, weights)))
        return F.gelu(self.norm2(self.sage2(x, edges, weights) + x))


def _pairs(rows, recipe_index, ingredient_index):
    if not rows:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    pairs = torch.tensor(
        [(recipe_index[recipe], ingredient_index[name]) for recipe, name, _ in rows],
        dtype=torch.long,
    ).t().contiguous()
    return pairs, torch.tensor([weight for _, _, weight in rows], dtype=torch.float)


def load_stage2(
    val_ratio: float,
    stage1_data: Path = STAGE1_DATA,
    stage1_checkpoint: Path = STAGE1_CHECKPOINT,
    stage2_data: Path = STAGE2_DATA,
) -> dict:
    """training/data.py load_dataset과 같은 노드 순서·학습/테스트 대상 + 고정된 1단계 레시피 임베딩

    val_ratio > 0이면 학습 분할에서 그 비율을 검증 쌍으로 떼어 낸다 (나머지 학습 쌍의 순서는 유지).
    검증 쌍은 학습 대상과 학습용 메시지 전달 그래프에서 모두 빠지고, 테스트 평가 그래프에는 들어간다
    (테스트 시점에는 아는 연결이므로). val_ratio=0이면 기존 2단계 학습·평가와 완전히 같다.
    """
    stage1 = torch.load(stage1_data, map_location="cpu", weights_only=True)
    stage2 = torch.load(stage2_data, map_location="cpu", weights_only=True)
    saved = torch.load(stage1_checkpoint, map_location="cpu", weights_only=True)
    if "artifact_sha256" in saved and saved["artifact_sha256"] != sha256(stage1_data):
        raise ValueError("stage-1 checkpoint was trained with a different data file")
    recipe_ids = [str(value) for value in stage1["recipe_ids"]]
    names = list(stage2["ingredient_names"])
    recipe_index = {value: index for index, value in enumerate(recipe_ids)}
    ingredient_index = {name: index for index, name in enumerate(names)}
    # 1단계 그래프에 없는 레시피의 기여도 행은 건너뜀 (load_dataset과 동일)
    rows = {
        split: [row for row in stage2["splits"][split] if row[0] in recipe_index]
        for split in ("train", "test")
    }
    selected = set(
        random.Random(VAL_SEED).sample(
            range(len(rows["train"])), round(len(rows["train"]) * val_ratio)
        )
    )
    train_rows = [row for i, row in enumerate(rows["train"]) if i not in selected]
    val_rows = [row for i, row in enumerate(rows["train"]) if i in selected]

    # 고정된 1단계 레시피 임베딩 (stage2 train의 recipe_model(data)와 같음)
    features = stage1["recipe_features"].float()
    encoder = FrozenRecipeEncoder(features.shape[1], saved["hidden"])
    encoder.load_state_dict(
        {k: v for k, v in saved["model"].items() if k.split(".")[0] in dict(encoder.named_children())}
    )
    encoder.eval()
    # CPU 단일 스레드로 계산해 실행 환경과 관계없이 같은 임베딩을 얻음
    threads = torch.get_num_threads()
    torch.set_num_threads(1)
    with torch.no_grad():
        recipe_embeddings = encoder(features, *stage1_context_edges(stage1["splits"]["train"]))
    torch.set_num_threads(threads)
    return {
        "recipe_embeddings": recipe_embeddings,
        "ingredient_features": stage2["ingredient_features"].float(),
        "train": _pairs(train_rows, recipe_index, ingredient_index),
        "val": _pairs(val_rows, recipe_index, ingredient_index),
        "test": _pairs(rows["test"], recipe_index, ingredient_index),
        # 테스트 평가용 메시지 전달 그래프: 원래 학습 분할 전체 (검증 쌍 포함)
        "test_graph": _pairs(rows["train"], recipe_index, ingredient_index),
    }


def without_pairs(edges, weights, pairs, width):
    # 예측 대상 연결을 메시지 전달 그래프에서 뺌 (training/stage2.py _without_pairs와 동일, 순서 유지)
    keys = edges[0] * width + edges[1]
    keep = ~torch.isin(keys, pairs[0] * width + pairs[1])
    return edges[:, keep], weights[keep]


def mean_absolute_error(weights, labels) -> float:
    if labels.numel() == 0:
        return float("nan")
    return float((weights - labels).abs().mean())


def root_mean_squared_error(weights, labels) -> float:
    if labels.numel() == 0:
        return float("nan")
    return float((weights - labels).square().mean().sqrt())


# ---------------- 모델 ----------------


class BipartiteAggregator(nn.Module):
    # 출발 노드 벡터를 도착 노드별로 모음 [도착 노드 수, hidden] (이웃이 없는 노드는 0 벡터)
    def __init__(self, kind: str, hidden: int, gru_order: str = "fixed"):
        super().__init__()
        if kind not in AGGREGATORS:
            raise ValueError(f"unknown aggregator: {kind}")
        if gru_order not in GRU_ORDERS:
            raise ValueError(f"unknown gru_order: {gru_order}")
        self.kind = kind
        self.gru_order = gru_order
        if kind == "max":
            self.pool = nn.Linear(hidden, hidden)
        elif kind == "gru":
            self.gru = nn.GRU(hidden + 1, hidden, batch_first=True)

    def forward(self, source, target, edges, weights, generator=None):
        if self.kind == "weighted_count":
            # 현재 2단계 WeightedSAGEConv (src/engine.py)와 같은 연산 순서: 가중합 / 이웃 수
            neighbors = torch.zeros_like(target)
            counts = target.new_zeros((target.shape[0], 1))
            if edges.numel():
                messages = source[edges[0]] * weights.unsqueeze(-1)
                neighbors.index_add_(0, edges[1], messages)
                counts.index_add_(0, edges[1], torch.ones_like(weights).unsqueeze(-1))
            return neighbors / counts.clamp_min(1)
        if not edges.numel():
            return torch.zeros_like(target)
        src, dst = edges
        if self.kind == "weighted_mean":
            # 1단계 방식: 가중합 / 가중치 합
            neighbors = torch.zeros_like(target).index_add(0, dst, source[src] * weights[:, None])
            totals = target.new_zeros(len(target)).index_add(0, dst, weights)
            return neighbors / totals.clamp_min(1e-8)[:, None]
        if self.kind == "mean":
            neighbors = torch.zeros_like(target).index_add(0, dst, source[src])
            counts = target.new_zeros(len(target)).index_add(0, dst, torch.ones_like(weights))
            return neighbors / counts.clamp_min(1)[:, None]
        if self.kind == "max":
            messages = F.relu(self.pool(source[src])) * weights[:, None]
            # 메시지가 모두 0 이상이므로 0에서 시작해도 이웃 최댓값과 같음
            return torch.zeros_like(target).scatter_reduce(
                0, dst[:, None].expand_as(messages), messages, reduce="amax"
            )
        return self._gru(source, target, src, dst, weights, generator)

    def _neighbor_order(self, dst, weights, generator):
        # 엣지를 도착 노드별로 모으고, 노드 안에서는 gru_order 순서로 놓는 순열
        if self.gru_order == "shuffle" and self.training:
            # 셔플 난수 생성기는 CPU에 두고 순열만 학습 장치로 옮김
            first = torch.randperm(len(dst), generator=generator).to(dst.device)
        elif self.gru_order == "weight":
            first = torch.argsort(weights, stable=True)
        else:  # fixed (shuffle 모델의 평가 포함): 엣지 목록 순서
            first = torch.arange(len(dst), device=dst.device)
        return first[torch.argsort(dst[first], stable=True)]

    def _gru(self, source, target, src, dst, weights, generator):
        order = self._neighbor_order(dst, weights, generator)
        src, dst, weights = src[order], dst[order], weights[order]
        nodes, counts = torch.unique_consecutive(dst, return_counts=True)
        starts = torch.cumsum(counts, 0) - counts
        rows = torch.repeat_interleave(torch.arange(len(nodes), device=dst.device), counts)
        positions = torch.arange(len(dst), device=dst.device) - torch.repeat_interleave(
            starts, counts
        )
        sequences = source.new_zeros(len(nodes), int(counts.max()), source.shape[1] + 1)
        sequences[rows, positions] = torch.cat((source[src], weights[:, None]), dim=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            sequences, counts.cpu(), batch_first=True, enforce_sorted=False
        )
        _, state = self.gru(packed)
        return torch.zeros_like(target).index_copy(0, nodes, state[-1])


class ContributionConv(nn.Module):
    # 2단계 WeightedSAGEConv에서 이웃 집계만 바꿀 수 있게 한 계층 (파라미터 이름·생성 순서 동일)
    def __init__(self, hidden: int, aggregator: str, gru_order: str):
        super().__init__()
        self.source = nn.Linear(hidden, hidden, bias=False)
        self.target = nn.Linear(hidden, hidden)
        self.aggregate = BipartiteAggregator(aggregator, hidden, gru_order)

    def forward(self, source, target, edges, weights, generator=None):
        neighbors = self.aggregate(source, target, edges, weights, generator)
        return self.target(target) + self.source(neighbors)


class ExperimentIngredientLinkEncoder(nn.Module):
    """IngredientLinkEncoder에서 집계 방식과 레이어 수만 바꿀 수 있게 한 실험용 모델

    레이어 l마다 (레시피 -> 재료) conv와 (재료 -> 레시피) reverse conv를 한 번씩 적용한다.
        재료_l   = conv_l(레시피_{l-1} -> 재료_{l-1})            (마지막 레이어가 아니면 ReLU)
        레시피_l = 레시피_{l-1} + reverse_conv_l(재료_{l-1} -> 레시피_{l-1})   (residual)
    num_layers=1이면 기존 모델과 같은 계산이다 (재료 입력 사영도 기존처럼 두 방향에서 따로 계산).
    num_layers=0이면 연결을 쓰지 않는 기준 모델(재료 특징 + 고정 레시피 임베딩만 사용)이다.
    """

    def __init__(
        self,
        ingredient_features: int,
        hidden: int,
        num_layers: int = 1,
        aggregator: str = "weighted_count",
        gru_order: str = "fixed",
    ):
        super().__init__()
        if num_layers < 0:
            raise ValueError("num_layers must be non-negative")
        self.input = nn.Linear(ingredient_features, hidden)
        # 기존 모델과 같은 순서(conv -> reverse_conv)로 만들어 같은 시드면 같은 초기값이 되게 함
        self.convs = nn.ModuleList()
        self.reverse_convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(ContributionConv(hidden, aggregator, gru_order))
            self.reverse_convs.append(ContributionConv(hidden, aggregator, gru_order))
        self.recipe_head = nn.Linear(hidden, hidden)
        self.ingredient_head = nn.Linear(hidden, hidden)
        self.logit_scale = nn.Parameter(torch.tensor(2.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def encode(self, ingredient_x, recipe_embeddings, edges, weights, generator=None):
        # edges: [2, 연결 수] (레시피 노드, 재료 노드), weights: 기여도 0~1
        reverse = edges.flip(0)
        ingredients = F.relu(self.input(ingredient_x))
        recipes = recipe_embeddings
        layers = len(self.convs)
        for index, (conv, reverse_conv) in enumerate(zip(self.convs, self.reverse_convs)):
            next_ingredients = conv(recipes, ingredients, edges, weights, generator)
            source = F.relu(self.input(ingredient_x)) if index == 0 else ingredients
            recipes = recipes + reverse_conv(source, recipes, reverse, weights.clone(), generator)
            ingredients = next_ingredients if index == layers - 1 else F.relu(next_ingredients)
        return F.normalize(recipes, dim=-1), F.normalize(ingredients, dim=-1)

    def score(self, ingredient_x, recipe_embeddings, edges, weights, pairs, generator=None):
        recipes, ingredients = self.encode(ingredient_x, recipe_embeddings, edges, weights, generator)
        recipe = F.normalize(self.recipe_head(recipes[pairs[0]]), dim=-1)
        ingredient = F.normalize(self.ingredient_head(ingredients[pairs[1]]), dim=-1)
        return (recipe * ingredient).sum(-1) * self.logit_scale.exp().clamp(max=100) + self.bias


# ---------------- 학습 (training/stage2.py train과 같은 절차) ----------------


@dataclass(frozen=True)
class Config:
    # 기본값 = 현재 2단계 설정
    aggregator: str = "weighted_count"
    gru_order: str = "fixed"
    num_layers: int = 1
    lr: float = 0.01
    epochs: int = 100  # 2단계는 조기 종료 없이 정해진 epoch만큼 학습하고 마지막 모델을 씀
    folds: int = 5
    val_ratio: float = 0.15
    seed: int = 42
    # 학습 장치도 설정에 포함: CPU/GPU 결과는 수치가 달라 이어 하기 캐시에서 섞이지 않게 함
    device: str = DEVICE


_DATASETS: dict[tuple, dict] = {}


def dataset(val_ratio: float) -> dict:
    # 경로는 호출 시점의 전역 값을 사용 (설정 셀에서 바꾼 경로 반영)
    key = (val_ratio, str(STAGE1_DATA), str(STAGE1_CHECKPOINT), str(STAGE2_DATA))
    if key not in _DATASETS:
        _DATASETS[key] = load_stage2(val_ratio, STAGE1_DATA, STAGE1_CHECKPOINT, STAGE2_DATA)
    return _DATASETS[key]


def train_once(config: Config, verbose: bool = False) -> dict:
    """설정 하나를 학습하고 검증/테스트 MAE·RMSE(+epoch별 기록)를 반환"""
    if config.epochs < 1 or config.lr <= 0 or config.folds < 1:
        raise ValueError("epochs, lr and folds must be positive")
    device = torch.device(config.device)
    if device.type == "cpu":
        torch.set_num_threads(1)
        torch.use_deterministic_algorithms(True)
    else:
        # 결정론적 구현이 없는 CUDA 연산은 오류 대신 경고 (해당 연산만 미세하게 달라질 수 있음)
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    data = dataset(config.val_ratio)
    recipe_embeddings = data["recipe_embeddings"].to(device)
    ingredient_x = data["ingredient_features"].to(device)
    width = len(ingredient_x)
    pairs, labels = (t.to(device) for t in data["train"])
    val_pairs, val_labels = (t.to(device) for t in data["val"])
    test_pairs, test_labels = (t.to(device) for t in data["test"])
    graph_edges, graph_weights = pairs, labels  # 학습용 메시지 전달 그래프 = 학습 쌍
    test_edges, test_weights = (t.to(device) for t in data["test_graph"])
    empty_edges = torch.empty((2, 0), dtype=torch.long, device=device)
    empty_weights = torch.empty(0, device=device)

    # 기존 stage2 train은 초기화 시드를 고정하지 않아 실행마다 달라짐: 여기서는 시드로 고정
    torch.manual_seed(config.seed)
    model = ExperimentIngredientLinkEncoder(
        ingredient_x.shape[1],
        recipe_embeddings.shape[1],
        config.num_layers,
        config.aggregator,
        config.gru_order,
    ).to(device)  # CPU에서 시드로 초기화한 뒤 옮기므로 초기 가중치는 장치와 무관하게 같음
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    generator = torch.Generator().manual_seed(config.seed)  # 폴드 나누기 (기존과 동일)
    neighbor_generator = torch.Generator().manual_seed(config.seed)  # GRU 이웃 셔플 전용

    def predict(edges, weights, target_pairs):
        model.eval()
        with torch.no_grad():
            return model.score(ingredient_x, recipe_embeddings, edges, weights, target_pairs).sigmoid()

    history = []
    loss = torch.tensor(float("nan"))
    # 폴드 교대: 현재 폴드의 정답 연결은 입력 그래프에서 제외하고 나머지 연결로 예측
    # 연결이 없는 재료도 평가할 수 있도록 연결을 모두 비운 그래프의 손실도 함께 학습
    for epoch in range(1, config.epochs + 1):
        model.train()
        order = torch.randperm(labels.numel(), generator=generator)
        losses = []
        for selected in order.chunk(min(config.folds, labels.numel())):
            selected = selected.to(device)
            fold_pairs, fold_labels = pairs[:, selected], labels[selected]
            context = without_pairs(graph_edges, graph_weights, fold_pairs, width)
            optimizer.zero_grad()
            context_loss = F.mse_loss(
                model.score(ingredient_x, recipe_embeddings, *context, fold_pairs, neighbor_generator).sigmoid(),
                fold_labels,
            )
            feature_loss = F.mse_loss(
                model.score(ingredient_x, recipe_embeddings, empty_edges, empty_weights, fold_pairs, neighbor_generator).sigmoid(),
                fold_labels,
            )
            loss = (context_loss + feature_loss) / 2
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
        record = {"epoch": epoch, "train_loss": statistics.fmean(losses)}
        if val_labels.numel():
            val_weights = predict(graph_edges, graph_weights, val_pairs)
            record["val_mae"] = mean_absolute_error(val_weights, val_labels)
            record["val_rmse"] = root_mean_squared_error(val_weights, val_labels)
        history.append(record)
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(record, flush=True)
    if not math.isfinite(float(loss)):
        raise ValueError("non-finite training loss")
    # 마지막 epoch 모델로 평가 (기존과 동일)
    # 검증: 학습 쌍만 있는 그래프, 테스트: 원래 학습 분할 전체(검증 쌍 포함) 그래프 = training/evaluate.py
    val_weights = predict(graph_edges, graph_weights, val_pairs)
    test_predictions = predict(test_edges, test_weights, test_pairs)
    best = min(history, key=lambda h: h.get("val_rmse", math.inf))
    return {
        "settings": asdict(config),
        "final_train_loss": float(loss.detach()),
        # 참고: 검증 RMSE가 가장 낮았던 epoch (학습은 조기 종료 없이 끝까지 진행)
        "best_val_epoch": best["epoch"] if "val_rmse" in best else None,
        "parameters": sum(p.numel() for p in model.parameters()),
        "val": {
            "n": val_labels.numel(),
            "mae": mean_absolute_error(val_weights, val_labels),
            "rmse": root_mean_squared_error(val_weights, val_labels),
        },
        "test": {
            "n": test_labels.numel(),
            "mae": mean_absolute_error(test_predictions, test_labels),
            "rmse": root_mean_squared_error(test_predictions, test_labels),
        },
        "history": history,
    }


# ---------------- 실험 반복 (조건 x 시드) 및 요약 ----------------

METRICS = ("val_rmse", "val_mae", "test_rmse", "test_mae", "final_train_loss")


def _flatten(label: str, report: dict) -> dict:
    return {
        "condition": label,
        "seed": report["settings"]["seed"],
        "best_val_epoch": report["best_val_epoch"],
        "parameters": report["parameters"],
        "val_rmse": report["val"]["rmse"],
        "val_mae": report["val"]["mae"],
        "test_rmse": report["test"]["rmse"],
        "test_mae": report["test"]["mae"],
        "final_train_loss": report["final_train_loss"],
    }


def run_grid(
    name: str,
    conditions: dict[str, dict],
    seeds: list[int],
    base: Config = Config(),
    root: Path = RESULTS,
) -> list[dict]:
    """조건(이름 -> Config에서 바꿀 값) x 시드를 모두 학습하고 결과 행 목록을 반환

    결과는 root/name/<조건>/seed<시드>/metrics.json에 저장하며, 같은 설정으로 이미 학습한
    결과가 있으면 다시 학습하지 않고 읽는다 (중간에 멈춰도 이어서 실행 가능).
    """
    rows = []
    total = len(conditions) * len(seeds)
    for label, overrides in conditions.items():
        for seed in seeds:
            config = replace(base, **overrides, seed=seed)
            out = Path(root) / name / label.replace("=", "_") / f"seed{seed}"
            path = out / "metrics.json"
            report = None
            if path.exists():
                saved = json.loads(path.read_text(encoding="utf-8"))
                if saved["settings"] == asdict(config):
                    report = saved
            if report is None:
                report = train_once(config)
                history = report.pop("history")
                out.mkdir(parents=True, exist_ok=True)
                path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
                (out / "history.json").write_text(
                    json.dumps(history, indent=2) + "\n", encoding="utf-8"
                )
            rows.append(_flatten(label, report))
            print(
                f"[{len(rows)}/{total}] {label} seed={seed} "
                f"val_rmse={report['val']['rmse']:.4f} val_mae={report['val']['mae']:.4f} "
                f"test_rmse={report['test']['rmse']:.4f}",
                flush=True,
            )
    _write_csv(Path(root) / name / "runs.csv", rows)
    summary = summarize(rows, conditions, base)
    _write_csv(
        Path(root) / name / "summary.csv",
        [{k: v for k, v in row.items() if k != "config"} for row in summary],
    )
    (Path(root) / name / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    return rows


def summarize(rows: list[dict], conditions: dict[str, dict], base: Config = Config()) -> list[dict]:
    # 조건별 시드 평균·표준편차 (검증 RMSE 평균 오름차순, 검증 쌍이 없으면 테스트 RMSE)
    summary = []
    for label, overrides in conditions.items():
        group = [row for row in rows if row["condition"] == label]
        if not group:
            continue
        entry = {
            "condition": label,
            "runs": len(group),
            # 시드·장치는 다음 실험에 넘기지 않음 (다음 실험은 현재 세션의 장치 사용)
            "config": {
                k: v
                for k, v in asdict(replace(base, **overrides)).items()
                if k not in ("seed", "device")
            },
            "device": base.device,
            "parameters": group[0]["parameters"],
        }
        for key in METRICS:
            values = [float(row[key]) for row in group]
            if any(math.isnan(value) for value in values):  # VAL_RATIO=0이면 검증 지표 없음
                entry[f"{key}_mean"] = entry[f"{key}_std"] = float("nan")
                continue
            entry[f"{key}_mean"] = statistics.fmean(values)
            entry[f"{key}_std"] = statistics.stdev(values) if len(values) > 1 else 0.0
        epochs = [row["best_val_epoch"] for row in group if row["best_val_epoch"] is not None]
        entry["best_val_epoch_mean"] = statistics.fmean(epochs) if epochs else float("nan")
        summary.append(entry)
    key = "val_rmse_mean" if base.val_ratio > 0 else "test_rmse_mean"
    return sorted(summary, key=lambda entry: entry[key])


def best_config(name: str, root: Path = RESULTS) -> tuple[str, dict]:
    """이전 실험 요약에서 1위 조건의 (이름, 설정) 반환"""
    summary = json.loads((Path(root) / name / "summary.json").read_text(encoding="utf-8"))
    return summary[0]["condition"], summary[0]["config"]


def markdown_table(summary: list[dict], noise: float | None = None) -> str:
    """요약 표 (평균 ± 표준편차). 1위와의 검증 RMSE 차이가 noise보다 작은 조건에 ≈ 표시 (시드 노이즈 수준)"""
    columns = (
        ("val_rmse", "검증 RMSE"),
        ("val_mae", "검증 MAE"),
        ("test_rmse", "테스트 RMSE"),
        ("test_mae", "테스트 MAE"),
        ("final_train_loss", "마지막 학습 손실"),
    )
    header = "| 순위 | 조건 | " + " | ".join(title for _, title in columns) + " | 검증 최저 epoch |"
    lines = [header, "|" + "---|" * (len(columns) + 3)]
    key = "val_rmse_mean" if summary and not math.isnan(summary[0]["val_rmse_mean"]) else "test_rmse_mean"
    best = summary[0][key] if summary else 0.0
    if noise is None and summary:
        noise = summary[0][key.replace("_mean", "_std")]  # 기본: 1위 조건의 시드 간 표준편차
    for rank, entry in enumerate(summary, 1):
        cells = [f"{entry[f'{k}_mean']:.4f} ± {entry[f'{k}_std']:.4f}" for k, _ in columns]
        mark = " ≈" if rank > 1 and entry[key] - best < noise else ""
        lines.append(
            f"| {rank} | {entry['condition']}{mark} | " + " | ".join(cells)
            + f" | {entry['best_val_epoch_mean']:.1f} |"
        )
    return "\n".join(lines)


def _write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def plot_summary(summary: list[dict], title: str) -> None:
    # 조건별 검증/테스트 RMSE 평균 ± 표준편차 (Colab 기본 폰트에 한글이 없어 그래프 글자는 영어)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib이 없어 그래프를 건너뜀")
        return
    labels = [entry["condition"] for entry in summary]
    positions = range(len(labels))
    fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(labels)), 4))
    for offset, part in ((-0.18, "val"), (0.18, "test")):
        ax.errorbar(
            [x + offset for x in positions],
            [entry[f"{part}_rmse_mean"] for entry in summary],
            yerr=[entry[f"{part}_rmse_std"] for entry in summary],
            fmt="o", capsize=4, label=part,
        )
    ax.set_xticks(list(positions), labels, rotation=20)
    ax.set_ylabel("RMSE (lower is better)")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_histories(name: str, labels: list[str], seed: int, root: Path = RESULTS) -> None:
    # 조건별 epoch에 따른 검증 RMSE (시드 하나)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib이 없어 그래프를 건너뜀")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    for label in labels:
        path = Path(root) / name / label.replace("=", "_") / f"seed{seed}" / "history.json"
        if not path.exists():
            continue
        history = json.loads(path.read_text(encoding="utf-8"))
        if "val_rmse" not in history[0]:
            continue
        ax.plot([h["epoch"] for h in history], [h["val_rmse"] for h in history], label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val RMSE")
    ax.set_title(f"{name}: learning curves (seed {seed})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ======================================================================
# [2] 공통 설정
# ======================================================================
from IPython.display import Markdown, display

SEEDS = [42, 43, 44, 45, 46]  # 조건마다 이 시드들로 반복 학습
VAL_RATIO = 0.15  # 학습 분할에서 검증으로 뗄 비율 (0이면 기존 2단계와 같은 절차, 조건 선택은 테스트로)
STAGE1_DATA = Path("data/stage1_rebuild_raw_gte.pt")
STAGE1_CHECKPOINT = Path("checkpoints/stage1_rebuild_raw_gte_h64.pt")
STAGE2_DATA = Path("tmp/stage2_embeddings.pt")
RESULTS = Path("results/experiments_stage2")

for path in (STAGE1_DATA, STAGE1_CHECKPOINT, STAGE2_DATA):
    if not path.exists():
        raise FileNotFoundError(f"{path}가 없습니다. Colab 파일 탭에서 /content/{path.parent}/에 올려 주세요.")
loaded = dataset(VAL_RATIO)
print("torch", torch.__version__, "| 장치", DEVICE, "| 시드", SEEDS)
print(
    f"기여도 쌍: 학습 {loaded['train'][1].numel()} / 검증 {loaded['val'][1].numel()} / "
    f"테스트 {loaded['test'][1].numel()} | 레시피 {len(loaded['recipe_embeddings'])}, "
    f"재료 {len(loaded['ingredient_features'])}, hidden {loaded['recipe_embeddings'].shape[1]}"
)

## 실험 1. 이웃 집계 방식

레이어 1, 학습률 0.01 등 나머지는 현재 설정 그대로 두고 두 방향(레시피 → 재료, 재료 → 레시피) conv의 **이웃 집계 부분만** 바꿉니다.

| 조건 | 집계 결과 | 엣지 가중치(기여도) 사용 |
|---|---|---|
| `weighted_count` | Σ wⱼxⱼ / 이웃 수 (**현재 2단계 방식**) | O |
| `weighted_mean` | Σ wⱼxⱼ / Σ wⱼ (1단계 방식) | O |
| `mean` | 이웃 벡터의 단순 평균 | X |
| `max` | maxⱼ( wⱼ · ReLU(W xⱼ + b) ) (GraphSAGE pooling) | O |
| `gru_fixed` | 이웃 시퀀스 [xⱼ, wⱼ]를 GRU로 읽은 마지막 은닉 상태, **순서 고정**(엣지 목록 순서) | O |
| `gru_shuffle` | 위와 같되 **학습 중 forward마다 노드별 이웃 순서를 새로 섞음**. 평가는 고정 순서 | O |

- 현재 방식은 가중합을 가중치 합이 아니라 **이웃 수**로 나눕니다. 그래서 기여도가 작은 이웃이 많은 노드는 집계 벡터의 크기가 작아집니다. `weighted_mean`과 비교하면 이 차이의 효과를 볼 수 있습니다.
- 재료 → 레시피 방향은 레시피당 재료가 7개 안팎이지만, 레시피 → 재료 방향은 소금처럼 수백 개 레시피에 들어가는 재료가 있어(최대 806) GRU가 매우 긴 시퀀스를 읽습니다. 그래서 GRU 조건은 다른 조건보다 수십 배 느립니다(CPU 기준 epoch당 약 7.5초).
- `gru_weight`(기여도 오름차순 고정 정렬)도 구현돼 있습니다. 비교하려면 아래 주석을 해제하세요.

In [ ]:
# ======================================================================
# 실험 1. 이웃 집계 방식
# ======================================================================
EXPERIMENT_1 = "01_aggregator"
BASE_1 = Config(val_ratio=VAL_RATIO)  # 현재 2단계 설정: 레이어 1, Adam lr 0.01, 100 epoch, 5-폴드
CONDITIONS_1 = {
    "weighted_count": {"aggregator": "weighted_count"},
    "weighted_mean": {"aggregator": "weighted_mean"},
    "mean": {"aggregator": "mean"},
    "max": {"aggregator": "max"},
    "gru_fixed": {"aggregator": "gru", "gru_order": "fixed"},
    "gru_shuffle": {"aggregator": "gru", "gru_order": "shuffle"},
    # "gru_weight": {"aggregator": "gru", "gru_order": "weight"},
}

rows_1 = run_grid(EXPERIMENT_1, CONDITIONS_1, SEEDS, BASE_1, RESULTS)
summary_1 = summarize(rows_1, CONDITIONS_1, BASE_1)
display(Markdown(markdown_table(summary_1)))
plot_summary(summary_1, "Stage 2 / Exp 1. neighbor aggregator")

## 실험 2. GraphSAGE 레이어 수

실험 1의 1위 집계 방식을 자동으로 가져와 레이어 수만 바꿉니다. 직접 정하려면 `BASE_2`를 바꾸세요(예: `BASE_2 = Config(aggregator="max", val_ratio=VAL_RATIO)`).

- 레이어 하나 = (레시피 → 재료) conv + (재료 → 레시피) reverse conv입니다. 레이어를 쌓으면 재료가 "같은 레시피에 들어가는 다른 재료" 정보까지 받습니다.
  - 재료ₗ = conv(레시피ₗ₋₁ → 재료ₗ₋₁), 마지막 레이어가 아니면 ReLU
  - 레시피ₗ = 레시피ₗ₋₁ + reverse_conv(재료ₗ₋₁ → 레시피ₗ₋₁) (residual)
- `L1`이 현재 모델과 같습니다. `LAYERS`에 `0`을 넣으면 연결을 쓰지 않는 기준 모델(재료 특징 + 고정 레시피 임베딩만 사용)이 되어 그래프 자체의 기여를 볼 수 있습니다.

In [ ]:
# ======================================================================
# 실험 2. GraphSAGE 레이어 수
# ======================================================================
EXPERIMENT_2 = "02_num_layers"
best_label_1, best_1 = best_config(EXPERIMENT_1, RESULTS)  # 실험 1의 1위 설정
BASE_2 = Config(**best_1)
LAYERS = [1, 2, 3]  # 0을 추가하면 연결 미사용 기준 모델
CONDITIONS_2 = {f"L{n}": {"num_layers": n} for n in LAYERS}
print(f"집계 방식: {BASE_2.aggregator} (gru_order={BASE_2.gru_order}) | 실험 1의 1위: {best_label_1}")

rows_2 = run_grid(EXPERIMENT_2, CONDITIONS_2, SEEDS, BASE_2, RESULTS)
summary_2 = summarize(rows_2, CONDITIONS_2, BASE_2)
display(Markdown(markdown_table(summary_2)))
plot_summary(summary_2, f"Stage 2 / Exp 2. number of layers ({BASE_2.aggregator})")

## 실험 3. 학습률 (Adam)

실험 2의 1위 설정(집계 방식·레이어 수)을 자동으로 가져와 학습률만 바꿉니다. 직접 정하려면 `BASE_3`을 바꾸세요.

- 2단계는 **조기 종료 없이 `EPOCHS`(기본 100)만큼 학습하고 마지막 모델**을 씁니다(기존과 동일). 같은 epoch 예산에서 학습률을 비교하는 셈이라, 낮은 학습률은 덜 수렴하고 높은 학습률은 과적합될 수 있습니다.
- 표의 "검증 최저 epoch"는 검증 RMSE가 가장 낮았던 epoch의 평균입니다(참고용). 이 값이 `EPOCHS`에 가까우면 아직 좋아지는 중이었다는 뜻이고, 훨씬 작으면 이후 과적합됐다는 뜻입니다. 마지막 그래프(첫 번째 시드의 학습 곡선)도 함께 보세요.

In [ ]:
# ======================================================================
# 실험 3. 학습률
# ======================================================================
EXPERIMENT_3 = "03_learning_rate"
best_label_2, best_2 = best_config(EXPERIMENT_2, RESULTS)  # 실험 2의 1위 설정
EPOCHS = 100  # 학습 epoch 수 (조기 종료 없음, 기존 2단계와 같은 100)
BASE_3 = replace(Config(**best_2), epochs=EPOCHS)
LRS = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
CONDITIONS_3 = {f"lr={lr:g}": {"lr": lr} for lr in LRS}
print(
    f"집계 방식: {BASE_3.aggregator} (gru_order={BASE_3.gru_order}) | "
    f"레이어: {BASE_3.num_layers} | 실험 2의 1위: {best_label_2}"
)

rows_3 = run_grid(EXPERIMENT_3, CONDITIONS_3, SEEDS, BASE_3, RESULTS)
summary_3 = summarize(rows_3, CONDITIONS_3, BASE_3)
display(Markdown(markdown_table(summary_3)))
plot_summary(summary_3, "Stage 2 / Exp 3. learning rate")
plot_histories(EXPERIMENT_3, list(CONDITIONS_3), seed=SEEDS[0], root=RESULTS)

## 결과 다운로드

`results/experiments_stage2/` 전체를 zip으로 묶어 내려받습니다(Colab 전용).

In [ ]:
import shutil

archive = shutil.make_archive("stage2_graph_experiments", "zip", RESULTS)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print(f"Colab이 아니므로 다운로드 대신 파일만 만듦: {archive}")